# MediDigit: Medical Record Digitiser
### MLCC Activity-2 Mini Project | 25MCASC303 Machine Learning in Cloud Computing
**Team:** ABHISHEK KUMAR (25MCAR0141), ADITHYA TEJ B M (25MCAR0142) and SURYA R M (25MCAR0164)

**Modules covered:**
- Module 1: Vision API (OCR) and Natural Language API (theory; implemented using PaddleOCR and Tesseract OCR, TrOCR for handwriting, and regex).

**Faculty:** Dr. N. Gobi, Associate Professor

## 1. Problem and scope
Doctors cannot access patient history across hospitals. Records are paper and scattered PDFs, so a doctor sees only the current visit.

This project narrows that problem to one step: digitise a scanned medical record and classify it. Upload a record image, read the text with OCR, extract key fields, and classify the record as Prescription, Lab Report, Discharge Summary, or Imaging Report.


**Patient flow:** open notebook, run all, upload the prescription in the Upload cell under Section 2, rest is automatic. Result is saved to that account's Google Drive, in `MyDrive/MediDigit`, as one formatted Excel sheet, `medidigit_records.xlsx`. No other output file is produced.


## 2. Setup: Colab only
Run this cell first. It installs Tesseract and PaddleOCR. Pillow is already in Colab.

In [1]:
!apt-get install -y tesseract-ocr > /dev/null 2>&1
!pip install -q pytesseract paddleocr

import sys
print(sys.version)


3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
TensorFlow: 2.21.0
scikit-learn: 1.6.1


In [ ]:
import sqlite3, datetime, re
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from PIL import Image
import pytesseract
IST = datetime.timezone(datetime.timedelta(hours=5, minutes=30))
# Rule-based record type. Module 1 only, no trained model.
RECORD_KEYWORDS = {
    "Prescription": ["prescription", "tablet", "tab", "capsule", "syrup", "dose", "mg", "twice daily", "once daily", "sos", "paracetamol", "metformin", "amoxicillin", "atorvastatin", "omeprazole", "before food", "after food"],
    "Lab Report": ["lab report", "laboratory", "pathology", "reference range", "mg/dl", "haemoglobin", "wbc", "fasting glucose", "creatinine", "total cholesterol", "tsh", "hba1c", "bilirubin", "albumin", "ast", "alt", "liver function", "lft"],
    "Discharge Summary": ["discharge summary", "discharged", "admission", "discharge date", "diagnosis", "admitted"],
    "Imaging Report": ["ultrasonography", "ultrasound", "x-ray", "xray", "mri", "ct scan", "imaging", "scan", "echo", "radiology"],
}
def classify_record(text):
    low = text.lower()
    scores = {label: sum(1 for w in words if re.search(r"\b" + re.escape(w) + r"\b", low)) for label, words in RECORD_KEYWORDS.items()}
    total = sum(scores.values())
    if total == 0: return "Not found", 0.0
    label = max(scores, key=lambda k: scores[k])
    return label, round(scores[label] / total, 2)
def extract_fields(text):
    m = re.search(r"\d{2}[-./]\d{2}[-./]\d{4}", text)
    date = m.group(0).replace(".", "-").replace("/", "-") if m else "Not found"
    med = re.search(r"(Paracetamol|Metformin|Amoxicillin|Atorvastatin|Omeprazole)[^\.]*", text)
    test = re.search(r"(Haemoglobin|WBC|Fasting Glucose|Creatinine|Total Cholesterol)[^\.]*", text)
    return {"date": date, "medicine_or_test": (med.group(0) if med else test.group(0)) if (med or test) else "Not found"}
def extract_patient_name(text):
    m = re.search(r"PATIENTS?\s*NAME\s*:?\s*([A-Za-z][A-Za-z .]{2,40})", text, re.I)
    if m:
        words = m.group(1).strip().strip(".").split()
        while words and words[-1].rstrip(".").upper() in ("REF", "BY", "DR"): words.pop()
        if words: return " ".join(words)
    m = re.search(r"Mr/Mrs/Miss\s+([A-Za-z][A-Za-z ]+?)(?:\s*\(|Age|of the|\n)", text, re.I)
    if m: return m.group(1).strip()
    m = re.search(r"\bPatient\s+([A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+)?)", text)
    if m and m.group(1).split()[0] not in ("Details", "Name", "ID"): return m.group(1)
    m = re.search(r"\b([A-Z][a-z]+ [A-Z]\. [A-Z][a-z]+)\b", text)
    if m: return m.group(1)
    for line in text.splitlines():
        if re.match(r"^[A-Z][a-z]+ [A-Z]\. [A-Z][a-z]+$", line.strip()): return line.strip()
    skip = {"Hospital", "Clinic", "Diagnostic", "Centre", "Center", "Laboratory", "Lab", "Report", "Doctor", "Dr", "Test", "Sample", "Collected", "Reference", "Investigation", "Certificate", "Prescription", "Discharge", "Imaging", "Medical", "Patient", "Details", "Name", "ID", "Age", "Sex", "Date", "Ref", "UHID", "Note", "Signature", "Health", "For", "Your", "Maximum", "General", "Road", "Complex", "Street", "Mumbai", "City"}
    for line in text.splitlines():
        line = line.strip().strip(".,;:'\"")
        if re.match(r"^[A-Z][a-z]+(?: [A-Z][a-z]+){1,2}$", line) and not any(w in line.split() for w in skip): return line
    return "Not found"
def ocr_tesseract(path):
    from PIL import ImageOps
    img = Image.open(path).convert("L")
    w, h = img.size
    img = ImageOps.autocontrast(img.resize((w * 2, h * 2), Image.LANCZOS))
    return pytesseract.image_to_string(img, config="--psm 6")
def ocr_paddle(path):
    from paddleocr import PaddleOCR
    if "_paddle_reader" not in globals():
        try: globals()["_paddle_reader"] = PaddleOCR(use_textline_orientation=True, lang="en")
        except TypeError: globals()["_paddle_reader"] = PaddleOCR(lang="en")
    reader = globals()["_paddle_reader"]
    try: result = reader.predict(path)
    except Exception: result = reader.ocr(path)
    lines = []
    for res in result or []:
        if isinstance(res, dict): lines.extend(res.get("rec_texts") or [])
        elif isinstance(res, (list, tuple)):
            for item in res:
                if isinstance(item, (list, tuple)) and len(item) > 1 and isinstance(item[1], (list, tuple)): lines.append(item[1][0])
    return "\n".join(lines)
def ocr_image(path):
    texts = {}
    try: texts["paddle"] = ocr_paddle(path); print("PaddleOCR words:", len(texts["paddle"].split()))
    except Exception as e: print("PaddleOCR failed, using Tesseract. Reason:", e)
    try: texts["tesseract"] = ocr_tesseract(path); print("Tesseract words:", len(texts["tesseract"].split()))
    except Exception as e: print("Tesseract failed. Reason:", e)
    texts = {k: v for k, v in texts.items() if v.strip()}
    if not texts: return None
    best = max(texts, key=lambda k: len(texts[k].split()))
    print("OCR engine used:", best)
    return texts[best]
def to_indian_datetime(s):
    try: return datetime.datetime.strptime(str(s), "%Y-%m-%d %H:%M").replace(tzinfo=datetime.timezone.utc).astimezone(IST).strftime("%d-%m-%Y, %I:%M %p")
    except Exception: return str(s)
def to_indian_date(s):
    try: return datetime.datetime.strptime(str(s), "%Y-%m-%d").strftime("%d-%m-%Y")
    except Exception: return str(s)
MEDS = ["Paracetamol", "Metformin", "Amoxicillin", "Atorvastatin", "Omeprazole"]
TESTS = ["Haemoglobin", "WBC", "Fasting Glucose", "Creatinine", "Total Cholesterol", "X-ray", "MRI", "CT scan", "Ultrasound", "Ultrasonography", "ECG", "TSH", "HbA1c", "Bilirubin", "Albumin", "GGTP", "Alkaline Phosphatase", "Total Protein", "Globulin"]
def extract_test(text):
    for t in TESTS:
        if t.lower() in text.lower(): return t
    m = re.search(r"\b(AST|ALT)\b", text)
    return m.group(1) if m else "Not found"
def extract_timing(text):
    parts = []
    for pat in (r"(twice daily|once daily|three times daily|1-0-1|1-1-1|0-1-0|\bOD\b|\bBD\b|\bTDS\b|every \d+ hours)", r"(before food|after food|before meals|after meals|on empty stomach)", r"for (\d+) days?"):
        m = re.search(pat, text, re.I)
        if m: parts.append(m.group(0))
    return ", ".join(parts) if parts else "Not found"
def extract_safety(text):
    up = text.upper()
    sos = "SOS/PRN medicine present, take only when symptoms appear." if ("SOS" in up or "PRN" in up) else "No SOS/PRN medicine noted."
    if extract_medicines(text) != "Not found": sos += " Cross-check strength and tablet count on the bill with the prescription before leaving the pharmacy."
    return sos
def extract_age(text):
    for pat in (r"\bAge\s*:?\s*(\d{1,3})\b", r"\b(\d{1,3})\s*(?:years|yrs|year)\b"):
        m = re.search(pat, text, re.I)
        if m: return m.group(1)
    return "Not found"
def extract_doctor(text):
    for line in str(text).splitlines():
        m = re.search(r"Ref\.?\s*By:?\s*(?:Dr\.?\s*:?\s*)?([A-Z][A-Za-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][A-Za-z]+)?)", line, re.I)
        if m: return "Dr. " + m.group(1).strip()
        m = re.search(r"Dr\.?\s+[A-Z]\.?\s*[A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+)?", line, re.I)
        if m: return m.group(0).strip()
    return "Not found"
def extract_hospital(text):
    kws = ["Hospital", "Clinic", "Diagnostic Centre", "Pathology Lab", "Medical Centre", "Laboratory"]
    generic = {"lab", "report", "prescription", "discharge", "summary", "imaging", "medical", "certificate", "test", "results", "result", "investigation"}
    for line in str(text).splitlines():
        for k in kws:
            m = re.search(re.escape(k), line, re.I)
            if m:
                words = re.sub(r"\s+", " ", line[:m.end()]).split()
                stripped = list(words)
                while stripped and stripped[0].strip(" .;,-").lower() in generic: stripped.pop(0)
                if len(stripped) < 2: stripped = words
                name = " ".join(stripped).strip(" .;,-")
                if len(name) > 4: return name[:60]
    return "Not found"
def extract_doctor_hospital(text):
    parts = [p for p in (extract_doctor(text), extract_hospital(text)) if p != "Not found"]
    return "; ".join(parts) if parts else "Not found"
def dedup(items):
    seen = []
    for p in items:
        if any(p.lower() in s.lower() for s in seen): continue
        seen = [s for s in seen if s.lower() not in p.lower()]
        seen.append(p)
    return seen
def _test_fragments(text):
    t = str(text); frags = []
    names = TESTS + ["AST", "ALT"]
    stop_words = {"Note", "Ref", "Dr", "Thanks", "Generated", "Clinical", "Interpretation", "Sample"}
    for name in names:
        for m in re.finditer(r"\b" + re.escape(name) + r"\b", t, re.I):
            rm = re.match(r"(?:\s*\([^)]{0,30}\))?\)?\s*:?\s*(?:[A-Za-z]+(?:\s+[A-Za-z]+)?\s+)?([\d.]+(?:\s*(?:mg/dL|U/L|g/dL|mU/L|%|mg|ml)\b)?(?:\s*(?:High|Low|Normal))?)?", t[m.end():])
            frag = m.group(0); has_flag = False
            if rm and rm.group(1):
                frag += rm.group(0)
                has_flag = bool(re.search(r"High|Low|Normal", rm.group(1)))
                if not has_flag:
                    tail = t[m.end() + len(rm.group(0)):m.end() + len(rm.group(0)) + 12]
                    tm = re.match(r"\s+([A-Za-z/]{2,10})(?![A-Za-z])", tail)
                    if tm and tm.group(1) not in names and tm.group(1) not in stop_words: frag += " " + tm.group(1)
            frag = re.sub(r"\s+", " ", frag.replace("(", "").replace(")", "")).strip(" .;,-")
            if len(frag) > len(name) + 1: frags.append(frag)
    return frags
def extract_lab_results(text):
    out = "; ".join(dedup(_test_fragments(text)))
    return out[:300] if out else "Not found"
TITLE_PAT = r"(ULTRASONOGRAPHY OF [A-Z ]+EYE|LIVER FUNCTION TEST\s*\(LFT\)|THYROID STIMULATING HORMON\s*\(TSH\)|HbA1c\s*\(GLYCOSYLATED HEMOGLOBIN\)|Medical Certificate|Discharge Summary|Imaging Report|Lab Report|Prescription)"
def extract_findings(text):
    t = str(text); parts = []
    title = re.search(TITLE_PAT, t, re.I)
    if title: parts.append(title.group(0).strip())
    find_kws = ["cataract", "haemorrhage", "hemorrhage", "thickened", "opaque", "appears normal", "no detectable abnormality", "acute pharyngitis", "unfit to work"]
    for sent in re.split(r"(?<=[.;])\s+|\n+", t):
        sent = re.sub(r"^[^A-Za-z0-9]+", "", sent.strip())
        if len(sent) < 5: continue
        if any(k in sent.lower() for k in find_kws):
            tm = re.search(TITLE_PAT, sent, re.I)
            if tm: sent = sent[tm.start():]
            if len(sent) > 200: sent = sent[:200].rsplit(" ", 1)[0]
            parts.append(re.sub(r"\s+", " ", sent).strip(" .;,-"))
    parts = [re.sub(r"\s+", " ", p).strip(" .;,-") for p in parts]
    parts = [p for p in parts if p]
    out = "; ".join(dedup(parts))
    return out[:300] if out else "Not found"
def extract_medicines(text):
    t = str(text); parts = []
    for med in MEDS:
        for m in re.finditer(r"\b" + re.escape(med) + r"\b", t, re.I):
            frag = m.group(0)
            rm = re.match(r"\s+(\d+\s*(?:mg|ml|g)\b)?\s*(tablet|tab|cap|capsule|syp|syrup|drops|injection)?", t[m.end():], re.I)
            if rm and rm.group(0).strip(): frag += rm.group(0)
            parts.append(re.sub(r"\s+", " ", frag).strip(" .;,-"))
    timing = extract_timing(t)
    if timing != "Not found": parts.append(timing)
    seen = []
    for p in parts:
        if p.lower() not in [s.lower() for s in seen]: seen.append(p)
    out = "; ".join(seen)
    return out[:300] if out else "Not found"
def extract_allergies(text):
    t = str(text)
    if re.search(r"no known allerg|\bNKA\b|\bNKDA\b", t, re.I): return "No known allergy"
    m = re.search(r"allerg(?:y|ic)\s*(?:to|:)?\s*([A-Za-z][A-Za-z ,]{1,40})", t, re.I)
    if m: return ("Allergic to " + m.group(1).strip(" ,")).strip()
    for sent in re.split(r"(?<=[.;])\s+|\n+", t):
        if "allerg" in sent.lower(): return re.sub(r"\s+", " ", sent).strip(" .;,-")[:120]
    return "Not found"
def extract_advice(text):
    t = str(text); parts = []
    fu = re.search(r"(follow[- ]?up[^.;\n]{0,40}|review (?:after|in)[^.;\n]{0,40}|come back[^.;\n]{0,40})", t, re.I)
    if fu: parts.append(re.sub(r"\s+", " ", fu.group(0)).strip(" .;,-"))
    dur = re.search(r"\d+\s*days?\s*(at\s*)?rest", t, re.I)
    if dur: parts.append(dur.group(0).strip())
    ref = re.search(r"refer(?:red)? to[^.;\n]{0,40}", t, re.I)
    if ref: parts.append(re.sub(r"\s+", " ", ref.group(0)).strip(" .;,-"))
    out = "; ".join(parts)
    return out if out else "Not found"
def export_formatted_excel(df, path):
    wb = Workbook(); ws = wb.active; ws.title = "Records"
    headers = ["ID", "Patient Name", "Age", "Record Date", "Record Type", "Doctor and Hospital", "Key Findings", "Medicines", "Allergies", "Lab Results", "Doctor Advice", "Safety Checks", "Confidence (%)"]
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    header_fill = PatternFill("solid", fgColor="1F4E79")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin = Side(style="thin", color="B0B0B0"); border = Border(left=thin, right=thin, top=thin, bottom=thin)
    for col, h in enumerate(headers, start=1):
        cell = ws.cell(row=1, column=col, value=h)
        cell.font = header_font; cell.fill = header_fill; cell.alignment = header_align; cell.border = border
    for r, row in enumerate(df.itertuples(index=False), start=2):
        rid, pname, rdate, rtype, confv, dtext, kfield, sfile, created = row
        values = [rid, pname, extract_age(dtext), to_indian_date(rdate), rtype, extract_doctor_hospital(dtext), extract_findings(dtext), extract_medicines(dtext), extract_allergies(dtext), extract_lab_results(dtext), extract_advice(dtext), extract_safety(dtext), float(confv)]
        for c, value in enumerate(values, start=1):
            cell = ws.cell(row=r, column=c, value=value)
            cell.font = Font(name="Calibri", size=11); cell.border = border
            cell.alignment = Alignment(horizontal="center", vertical="top") if c in (1, 3, 4, 5, 13) else Alignment(vertical="top", wrap_text=True)
            if c == 13: cell.number_format = "0%"
    # AutoFit at export time, .xlsx only (a live macro needs .xlsm). Columns fit to the longest line (cap 50), rows to wrapped lines (cap 150).
    def disp(r, c):
        v = ws.cell(row=r, column=c).value
        return f"{v:.0%}" if c == 13 and isinstance(v, (int, float)) else str(v or "")
    col_width = {}
    for c in range(1, len(headers) + 1):
        longest = max(len(disp(r, c)) for r in range(1, len(df) + 2))
        col_width[c] = min(50, max(longest + 2, len(headers[c - 1]) + 4))
        ws.column_dimensions[get_column_letter(c)].width = col_width[c]
    for r in range(1, len(df) + 2):
        lines = 1
        for c in range(1, len(headers) + 1):
            for part in disp(r, c).split("\n"):
                lines = max(lines, max(1, -(-len(part) // max(1, int(col_width[c])))))
        ws.row_dimensions[r].height = min(150, max(30 if r == 1 else 15, lines * 15))
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{len(df) + 1}"
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.orientation = "landscape"; ws.page_setup.fitToWidth = 1; ws.page_setup.fitToHeight = 0
    wb.save(path)
def trocr_lines(path):
    import numpy as np, cv2
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    dilated = cv2.dilate(bw, cv2.getStructuringElement(cv2.MORPH_RECT, (40, 3)), iterations=1)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 60 and 8 < h < 120: boxes.append((x, y, w, h))
    boxes.sort(key=lambda b: (b[1], b[0]))
    lines = []
    for (x, y, w, h) in boxes:
        pad = 6
        x0 = max(0, x - pad); y0 = max(0, y - pad)
        x1 = min(img.shape[1], x + w + pad); y1 = min(img.shape[0], y + h + pad)
        lines.append(Image.fromarray(cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_BGR2RGB)))
    return lines
def trocr_read(path):
    try: from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    try:
        if "_trocr_model" not in globals():
            globals()["_trocr_processor"] = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
            globals()["_trocr_model"] = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
        proc, model = globals()["_trocr_processor"], globals()["_trocr_model"]
        try: line_imgs = trocr_lines(path)
        except Exception as e: print("Line detection failed, using whole page. Reason:", e); line_imgs = []
        if not line_imgs:
            img = Image.open(path).convert("RGB"); w, h = img.size
            line_imgs = [img.resize((w * 2, h * 2), Image.LANCZOS)]
        print("TrOCR lines:", len(line_imgs))
        texts = []
        for line_img in line_imgs:
            pixel_values = proc(line_img, return_tensors="pt").pixel_values
            texts.append(proc.batch_decode(model.generate(pixel_values), skip_special_tokens=True)[0])
        return "\n".join(t for t in texts if t.strip())
    except Exception as e:
        print("TrOCR failed, keep printed-text OCR result. Reason:", e)
        return ""
def save_record(text, label, conf, source_file):
    conn = sqlite3.connect(DB_PATH); cur = conn.cursor()
    cur.execute("CREATE TABLE IF NOT EXISTS records (id INTEGER PRIMARY KEY AUTOINCREMENT, patient_name TEXT, record_date TEXT, record_type TEXT, confidence REAL, digitised_text TEXT, key_field TEXT, source_file TEXT, created_at TEXT)")
    conn.commit()
    pname = extract_patient_name(text); fields = extract_fields(text)
    cur.execute("INSERT INTO records (patient_name, record_date, record_type, confidence, digitised_text, key_field, source_file, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?)", (pname, fields.get("date", "Not found"), label, float(conf), text, fields.get("medicine_or_test", "Not found"), source_file, datetime.datetime.now(IST).strftime("%d-%m-%Y, %I:%M %p")))
    conn.commit()
    df = pd.read_sql_query("SELECT * FROM records ORDER BY id", conn)
    export_formatted_excel(df, RECORDS_XLSX_PATH)
    print("Saved 1 record for:", pname)
    print("Output saved to Drive:", RECORDS_XLSX_PATH, "rows:", len(df))
    print("Total records in DB:", cur.execute("SELECT COUNT(*) FROM records").fetchone()[0])
    conn.close(); globals()["_saved_text"] = text
    return pname

# Auto-follow the executing cell: after this cell runs, Run All scrolls to each
# cell as it runs, so you watch it step by step with no clicking. No effect on
# the data or the Excel output. Runs once per session.
from IPython.display import Javascript, display
display(Javascript("""
if (!window._medidigitAutoFollow) {
  window._medidigitAutoFollow = true;
  setInterval(function() {
    var els = document.querySelectorAll('div');
    for (var i = 0; i < els.length; i++) {
      var el = els[i];
      if (el.children.length === 0 && el.textContent.trim() === '[*]') {
        var cell = el.closest('.cell');
        if (cell) cell.scrollIntoView({block: 'center', behavior: 'smooth'});
        break;
      }
    }
  }, 600);
}
"""))


## Upload your prescription here
Upload a prescription, lab report, discharge summary, or imaging report image. Tick the box in the next cell if it is handwritten. The row is saved to Drive automatically, no need to scroll down or run the cells below.


In [ ]:
# @title Upload your prescription here
handwritten = False # @param {type:"boolean"}

import os
DRIVE_FOLDER = "/content/drive/MyDrive/MediDigit"
DB_PATH = os.path.join(DRIVE_FOLDER, "medidigit_records.db")
RECORDS_XLSX_PATH = os.path.join(DRIVE_FOLDER, "medidigit_records.xlsx")
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    DRIVE_MOUNTED = True
    print("Drive mounted.")
    print("Auto-save folder:", DRIVE_FOLDER)
except Exception as e:
    print("Drive mount skipped, using local Colab folder. Reason:", e)
    DRIVE_FOLDER = "/content/MediDigit"
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    DB_PATH = os.path.join(DRIVE_FOLDER, "medidigit_records.db")
    RECORDS_XLSX_PATH = os.path.join(DRIVE_FOLDER, "medidigit_records.xlsx")
    print("Local fallback folder:", DRIVE_FOLDER)
    print("Warning: files here are lost on disconnect. Mount Drive to keep them.")
print("DB path:", DB_PATH)
print("Output Excel path:", RECORDS_XLSX_PATH)

uploaded = {}
try:
    from google.colab import files
    uploaded = files.upload()
except Exception as e: print("No file uploaded. Reason:", e)
if not uploaded:
    print("No file uploaded, nothing saved. Upload a file and re-run this cell.")
else:
    for fname in uploaded:
        print("Uploaded:", fname)
        uploaded_text = ocr_image(fname)
        if uploaded_text is None:
            print("OCR gave no text for", fname, ", skipped.")
            continue
        trocr_text = ""
        if handwritten:
            print("Handwritten ticked, running TrOCR (slow on CPU, a few minutes).")
            trocr_text = trocr_read(fname)
            if trocr_text.strip() and len(trocr_text.split()) > len(uploaded_text.split()):
                uploaded_text = trocr_text
                print("Stored text switched to TrOCR, it read more words.")
        pred_label, conf = classify_record(uploaded_text)
        print(f"Predicted type: {pred_label}  confidence: {conf:.2f}")
        print("Fields:", extract_fields(uploaded_text))
        save_record(uploaded_text, pred_label, conf, fname)
    _top_saved = True


## 2a. Auto-save to Google Drive: run once, works in any Colab account
This cell mounts the Google Drive of whoever runs the notebook, and creates the same folder every time. No email is hardcoded. The patient only uploads the prescription sheet later. Everything else is saved to Drive automatically.

- Drive folder: `MyDrive/MediDigit`
- Output, the only output file: `MyDrive/MediDigit/medidigit_records.xlsx`, a formatted Excel sheet with all digitised records, Indian date-time, confidence in %, and the 13 doctor-check columns: patient name and age, record date and type, doctor and hospital, key findings, medicines, allergies, lab results, doctor advice, safety checks, confidence.
- Working file, not output: database `medidigit_records.db`.

If Drive mount is skipped, the notebook falls back to a local Colab folder and warns you. In that case files are lost on disconnect, so mount Drive for the real run.


In [2]:
import os
DRIVE_FOLDER = "/content/drive/MyDrive/MediDigit"
DB_PATH = os.path.join(DRIVE_FOLDER, "medidigit_records.db")
RECORDS_XLSX_PATH = os.path.join(DRIVE_FOLDER, "medidigit_records.xlsx")
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    DRIVE_MOUNTED = True
    print("Drive mounted.")
    print("Auto-save folder:", DRIVE_FOLDER)
except Exception as e:
    print("Drive mount skipped, using local Colab folder. Reason:", e)
    DRIVE_FOLDER = "/content/MediDigit"
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    DB_PATH = os.path.join(DRIVE_FOLDER, "medidigit_records.db")
    RECORDS_XLSX_PATH = os.path.join(DRIVE_FOLDER, "medidigit_records.xlsx")
    print("Local fallback folder:", DRIVE_FOLDER)
    print("Warning: files here are lost on disconnect. Mount Drive to keep them.")
print("DB path:", DB_PATH)
print("Output Excel path:", RECORDS_XLSX_PATH)


Mounted at /content/drive
Drive mounted.
Auto-save folder: /content/drive/MyDrive/MediDigit
DB path: /content/drive/MyDrive/MediDigit/medidigit_records.db
Dataset path: /content/drive/MyDrive/MediDigit/medidigit_dataset.csv
Model path: /content/drive/MyDrive/MediDigit/medidigit_best.h5
Output Excel path: /content/drive/MyDrive/MediDigit/medidigit_records.xlsx


## 3. OCR demo: digitise a record image in Colab
This cell creates a sample record image from text with Pillow, then reads it back with Tesseract. No external file needed. For your own file, use the upload cell below.

In [12]:
from PIL import Image, ImageDraw
import pytesseract
sample_text = "Lab Report City General Hospital Date 06-11-2026 Patient Aarav Sharma Laboratory test result Fasting Glucose 98 mg/dL Reference range normal."
img = Image.new("RGB", (900, 220), color="white")
ImageDraw.Draw(img).multiline_text((20, 20), sample_text, fill="black")
img.save("sample_lab_report.png")
print("Saved: sample_lab_report.png")
print("OCR text:")
print(pytesseract.image_to_string(Image.open("sample_lab_report.png")))


Saved: sample_lab_report.png
OCR text:
Lab Report City General Hospital Date 08-11-2026 Patient Aarav Sharma Laboratory tet result Fasting Glucose 88 mgidlL Reference range normal



## 4. Demo: upload your own record, or use the sample
In Colab, run this cell and choose a synthetic record image or PDF page image. In this local run the cell falls back to the sample image above, so the notebook still runs end to end without a manual upload.

In [13]:
if globals().get("_top_saved"): print("Using the file uploaded at the top:", fname)
else:
    uploaded_text = None
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded: fname = list(uploaded.keys())[0]; print("Uploaded:", fname); uploaded_text = ocr_image(fname)
    except Exception as e: print("No Colab upload, using sample image. Reason:", e)
    if uploaded_text is None: uploaded_text = ocr_image("sample_lab_report.png")
print("Digitised text:")
print(uploaded_text)
pred_label, conf = classify_record(uploaded_text)
print(f"Predicted type: {pred_label}  confidence: {conf:.2f}")
print("Fields:", extract_fields(uploaded_text))
print("Note: classification only. This is not a diagnosis.")


Saving sample-14.jpg to sample-14.jpg
Uploaded: sample-14.jpg
Digitised text:
; .
n> Bay

wa’ | &
wi 0 _aftkind

i ee il oe
oer

O Degen
ip eee anily PE ood “ot
urine aducee

O foe
4 ml af otf stom

on

nek) ofl

re ee Han he

ah

A ah ie

v-2 [don ;

rar wit by yer fe
BB viranvetctinne

. ‘Ghorama animal for happier & healthier life. aad
& Taking care of your anima 1 sxagum Ron, Comat a

Predicted type: Prescription  confidence: 0.38
Fields: {'date': 'Not found', 'medicine_or_test': 'Not found'}
Note: classification only. This is not a diagnosis.


## 4a. Handwriting OCR with TrOCR: compare with printed-text OCR
Tesseract is printed-text OCR, so doctor cursive fails. TrOCR is a transformer model trained for handwriting, `microsoft/trocr-base-handwritten`, and it runs in Colab with no GCP key.

- First run downloads the model, about 500 MB, 2 to 3 minutes. Run this cell once, then it is cached.
- The cell splits the page into text lines first, TrOCR reads one line at a time, so a full-page prescription reads better. If TrOCR reads more words than the printed-text OCR, its text is the one stored.
- If TrOCR fails to load, the cell prints the reason and keeps the Tesseract result, so the notebook still runs.


In [14]:
!pip install -q transformers
img_path = globals().get("fname", "sample_lab_report.png")
print("TrOCR image:", img_path)
trocr_text = trocr_read(img_path)
print("TrOCR text:")
print(trocr_text)


TrOCR image: sample-14.jpg


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

... (4 outputs left out)

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1969: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


TrOCR text:
0 1


In [15]:
print("Best OCR text:"); print(uploaded_text)
print("\nTrOCR text:"); print(trocr_text if trocr_text else "No TrOCR output.")
if trocr_text.strip():
    t_label, t_conf = classify_record(trocr_text)
    print(f"\nTrOCR predicted type: {t_label} confidence: {t_conf:.2f}")
    print("TrOCR fields:", extract_fields(trocr_text))
    print(f"Tesseract predicted type: {pred_label} confidence: {conf:.2f}")
    print("Best OCR fields:", extract_fields(uploaded_text))
else: print("TrOCR gave no text, use Tesseract result.")
if trocr_text.strip() and len(trocr_text.split()) > len(uploaded_text.split()):
    uploaded_text = trocr_text; pred_label, conf = classify_record(uploaded_text)
    print("\nStored text switched to TrOCR, it read more words.")
    print(f"Stored type: {pred_label} confidence: {conf:.2f}")
else: print("\nStored text stays with printed-text OCR.")


Tesseract text:
; .
n> Bay

wa’ | &
wi 0 _aftkind

i ee il oe
oer

O Degen
ip eee anily PE ood “ot
urine aducee

O foe
4 ml af otf stom

on

nek) ofl

re ee Han he

ah

A ah ie

v-2 [don ;

rar wit by yer fe
BB viranvetctinne

. ‘Ghorama animal for happier & healthier life. aad
& Taking care of your anima 1 sxagum Ron, Comat a


TrOCR text:
0 1

TrOCR predicted type: Prescription confidence: 0.26
TrOCR fields: {'date': 'Not found', 'medicine_or_test': 'Not found'}
Tesseract predicted type: Prescription confidence: 0.38
Tesseract fields: {'date': 'Not found', 'medicine_or_test': 'Not found'}


## 5. Storage: output is one formatted Excel sheet in Google Drive
Patient flow, nothing manual after upload:
1. Patient opens this notebook in any Colab account, runs all cells, Drive mounts and asks for that account login.
2. Patient uploads the prescription white sheet in Section 4.
3. Notebook does OCR, classification, extraction, and saves the row automatically.
4. Next doctor opens the same Excel sheet in Drive, `MyDrive/MediDigit/medidigit_records.xlsx`, and sees all past records. No download, no other output file.

The Excel sheet is the output. Columns: ID, Patient Name, Age, Record Date, Record Type, Doctor and Hospital, Key Findings, Medicines, Allergies, Lab Results, Doctor Advice, Safety Checks, Confidence (%). Dates and times are Indian format (DD-MM-YYYY, 12-hour IST). Confidence is shown in percentages. Each column holds what a doctor checks in a previous record: identity and date, who wrote it and where, what was found, medicines with dose and timing, allergies, lab values with High/Low flags, what was advised, and safety checks. Where a detail is not stated in the record the cell says Not found, nothing is invented. Lab trend over visits comes from the patient history lookup, not from a single row. Patient name is Not found when the name cannot be read from the image. The `.db` file stays in Drive only as internal storage, so history survives disconnect. It is not an output file, the doctor never needs to open it.


In [16]:
demo_text = globals().get("uploaded_text", sample_text)
demo_label = globals().get("pred_label", "Lab Report")
demo_conf = float(globals().get("conf", 0.0))
source_file = globals().get("fname", "sample_lab_report.png")
if globals().get("_saved_text") == demo_text: print("Already saved by the upload cell at the top, skipping duplicate save.")
else: save_record(demo_text, demo_label, demo_conf, source_file)


Saved 1 record for: Not found
Output saved to Drive: /content/drive/MyDrive/MediDigit/medidigit_records.xlsx rows: 13
Total records in DB: 13


In [17]:
import sqlite3
conn = sqlite3.connect(DB_PATH); cur = conn.cursor()
search_name = "Aarav Sharma"
rows = cur.execute("SELECT record_date, record_type, confidence, key_field FROM records WHERE patient_name = ? ORDER BY record_date DESC", (search_name,)).fetchall()
history = pd.DataFrame(rows, columns=["date", "type", "confidence", "key_field"])
print(f"History for {search_name}: {len(history)} record(s)")
print(history.to_string(index=False))
latest = cur.execute("SELECT digitised_text FROM records WHERE patient_name = ? ORDER BY id DESC LIMIT 1", (search_name,)).fetchone()
print("\nLatest digitised text:")
print(latest[0] if latest else "No record found.")
conn.close()
print("\nDoctor opens in Drive:", RECORDS_XLSX_PATH)


History for Aarav Sharma: 7 record(s)
      date         type  confidence                                          key_field
 Not found Prescription    0.301111                                          Not found
 Not found   Lab Report    0.295054                                          Not found
 Not found Prescription    0.302427                                          Not found
 Not found Prescription    0.344418                                          Not found
 Not found   Lab Report    0.659384                                          Not found
08-11-2026   Lab Report    0.750658 Fasting Glucose 88 mgidlL Reference range normal\n
08-11-2026   Lab Report    0.814805 Fasting Glucose 88 mgidlL Reference range normal\n

Latest digitised text:
DRLOGY PATHOLOGY LAB X 0123456789 | 0912345678
& accurate | Caring | Instant [© drlogypathlab@drlogy.com
105-108, MART VISION COMPLEX, HEALTHCARE ROAD, OPPOSITE HEALTHCARE COMPLEX. MUMBAI- 689578

NANG

Sample Collected At:
125, Shiv complex,

### Output check: one Excel file in Drive
This cell only verifies the output Excel sheet is in Drive. The patient does nothing here.


In [18]:
import os
print("Drive mounted:", DRIVE_MOUNTED)
if os.path.exists(RECORDS_XLSX_PATH): print("Output found:", RECORDS_XLSX_PATH, os.path.getsize(RECORDS_XLSX_PATH), "bytes")
else: print("Output missing, run the earlier cells first:", RECORDS_XLSX_PATH)
if not DRIVE_MOUNTED:
    print("\nWarning: Drive was not mounted, so the Excel file is only in local Colab storage.")
    print("Re-run the Drive mount cell, then re-run the save cells.")
else: print("\nDone. Next doctor opens medidigit_records.xlsx in Drive, no other output file.")


Drive mounted: True
Output found: /content/drive/MyDrive/MediDigit/medidigit_records.xlsx 6956 bytes

Done. Next doctor opens medidigit_records.xlsx in Drive, no other output file.


## 6. Results, limits, and conclusion
Module 1 only. The notebook digitises a record image with OCR, extracts key fields with regex, and classifies the record type with keyword rules. No trained model, no tuning.

**How it works:** PaddleOCR reads printed text first, preprocessed Tesseract is the fallback, TrOCR reads handwriting line by line. The text with more words is stored. Regex extracts the date, medicines or tests, doctor and hospital, and other fields. Keyword rules classify the record as Prescription, Lab Report, Discharge Summary, or Imaging Report, and the confidence is the winning type's share of all matched keywords.

**OCR on printed sample, Section 4:** Tesseract read the sample lab report, date 08-11-2026, Fasting Glucose 88 mg/dL. It misread 06 as 08, test as tet, and mg/dL as mgidlL. Keyword rules classify the sample text as Lab Report, confidence 0.83.

**TrOCR, Section 4a:** TrOCR (microsoft/trocr-base-handwritten) reads handwriting line by line. On printed text it gives garbled text, which is expected, it is a handwriting model.

**Real-world limit, from the demo in Section 4:** a real handwritten prescription image from Manila Doctors Hospital was uploaded. OCR text was garbled, and date and medicine fields were Not found. Handwriting and poor scans remain the hard case. The notebook classifies record type only, it does not diagnose, and it must not be used for clinical decisions.

**Output:** the only output is one formatted Excel sheet, `MyDrive/MediDigit/medidigit_records.xlsx`, saved to Drive after every record. Dates and times are Indian format (DD-MM-YYYY, 12-hour IST), confidence is shown in percentages. The sheet has 13 columns, one for each thing a doctor checks in a previous record: ID, Patient Name, Age, Record Date, Record Type, Doctor and Hospital, Key Findings, Medicines, Allergies, Lab Results, Doctor Advice, Safety Checks, Confidence (%). Where a detail is not stated in the record the cell says Not found, nothing is invented. Patient name is Not found when the name cannot be read from the image, it is never filled in from another record.

**Conclusion:** a Colab-only Module 1 digitiser is feasible for the Activity-2 scope, with Drive auto-save so history survives disconnect. The next step, if faculty allows it later, is to replace Tesseract with Vision API and regex with Natural Language API, and to test on a larger, properly consented dataset.
